# Shooting method for the RWA two-mode model

Goal: find a trajectory (an "instanton"/MPEP connection) between two fixed points of the
doubled (classical + conjugate-momentum) phase space `(A, B, P_A, P_B)`, using
linearization around each fixed point to set up boundary conditions for `scipy.solve_bvp`.

**Where this notebook ended up.** A *direct* connection between the two classical pointer
states `fp_init = (+A0, 0, 0, 0)` and `fp_fin = (-A0, 0, 0, 0)` turns out to be numerically
degenerate (see the "Why the direct connection is hard" section below) — `solve_bvp` happily
"converges" to a trivial trajectory that never leaves `fp_init`. Instead, this notebook builds
a genuine, verified connection between `fp_init` and its momentum-space counterpart
`fp_swap = (0, 0, +A0, 0)`, i.e. from `A != 0, P_A = 0` to `A = 0, P_A != 0`.


In [ ]:
import numpy as np
from scipy.special import jv
from scipy.integrate import solve_bvp
import matplotlib.pyplot as plt


In [ ]:
hbar = 1
e = 1
phi_quantum = 1 / 2

#### Characterization of the circuit ####
w_a = 25.338776456203686
w_b = 2 * w_a

phi_a = 0.11
phi_b = 0.204
E_J = 37.12 * 2 * np.pi

# Environment
kappa_b = 5 / 10.4  # 1/ns, from Leon's paper
eps_p = 0.05
g = np.sin(eps_p) * E_J * phi_a**2 * phi_b       # (Leon's paper)
g2 = jv(1, eps_p) * E_J * phi_a**2 * phi_b        # (Leon's paper)
eps_d = 10.12 * g2

sqrt2 = np.sqrt(2)

# --- Classical (P_A = P_B = 0) fixed points of the two-mode system ---
A0 = np.sqrt(eps_d / g)
fp_init = np.array([A0, 0.0, 0.0, 0.0])            # pointer state 1
fp_fin = np.array([-A0, 0.0, 0.0, 0.0])            # pointer state 2
fp_saddle = np.array([0.0, -sqrt2 * eps_d / kappa_b, 0.0, 0.0])   # classical barrier top

# --- Momentum-space counterpart of fp_init: same "radius" A0^2 = A^2 + P_A^2, but living
#     entirely in the (P_A, P_B) plane instead of the (A, B) plane. This is also an exact
#     fixed point of the full nonlinear system (checked below). ---
fp_swap = np.array([0.0, 0.0, A0, 0.0])
fp_swap_neg = np.array([0.0, 0.0, -A0, 0.0])

print(f"g       = {g:.6f}")
print(f"g2      = {g2:.6f}")
print(f"eps_d   = {eps_d:.6f}")
print(f"kappa_b = {kappa_b:.6f}")
print()
print(f"fp_init    = {fp_init}")
print(f"fp_fin     = {fp_fin}")
print(f"fp_saddle  = {fp_saddle}")
print(f"fp_swap    = {fp_swap}")



\begin{equation}
H= (-\sqrt 2 g AB)P_A+[gA^2/\sqrt 2-\varepsilon_d/\sqrt 2 - \kappa_b B/2]P_B - \kappa_b P_B^2/2 +gP_A^2P_B/\sqrt 2
\end{equation}


In [ ]:
def vf(y, g, eps_d, kappa_b):
    """Equations of motion dy/dt for y = (A, B, P_A, P_B)."""
    A, B, PA, PB = y
    dA  = -sqrt2 * g * (A * B - PA * PB)
    dB  = -eps_d / sqrt2 + g * (A**2 + PA**2) / sqrt2 - kappa_b * (B / 2 + PB)
    dPA = -sqrt2 * g * (A * PB - PA * B)
    dPB =  sqrt2 * g * A * PA + kappa_b * PB / 2
    return np.array([dA, dB, dPA, dPB])


def jac(y, g, kappa_b):
    """Analytic Jacobian d(vf)/dy, evaluated at a single point y."""
    A, B, PA, PB = y
    c = sqrt2 * g
    return np.array([
        [-c * B,      -c * A,     c * PB,  c * PA],
        [ c * A, -kappa_b / 2,    c * PA, -kappa_b],
        [-c * PB,     c * PA,     c * B,  -c * A],
        [ c * PA,          0,     c * A,  kappa_b / 2],
    ])


def fun_jac(t, y, g, kappa_b):
    """Vectorized Jacobian for solve_bvp: shape (4, 4, n_points)."""
    A, B, PA, PB = y
    c = sqrt2 * g
    m = y.shape[1]
    zeros, ones = np.zeros(m), np.ones(m)
    return np.array([
        [-c * B,          -c * A,        c * PB,   c * PA],
        [ c * A,  -kappa_b / 2 * ones,   c * PA,  -kappa_b * ones],
        [-c * PB,          c * PA,       c * B,   -c * A],
        [ c * PA,          zeros,        c * A,    kappa_b / 2 * ones],
    ])


def H_of(y, g, eps_d, kappa_b):
    """Conserved Hamiltonian of the doubled system (should be ~0 on any trajectory
    connecting two fixed points that both have H = 0)."""
    A, B, PA, PB = y
    return (-sqrt2 * g * A * B * PA
            + (g * A**2 / sqrt2 - eps_d / sqrt2 - kappa_b * B / 2) * PB
            - kappa_b * PB**2 / 2
            + g * PA**2 * PB / sqrt2)


# sanity check: fp_init, fp_fin, fp_saddle, fp_swap should all be genuine fixed points
for name, fp in [('fp_init', fp_init), ('fp_fin', fp_fin),
                  ('fp_saddle', fp_saddle), ('fp_swap', fp_swap)]:
    print(f"vf({name:10s}) = {vf(fp, g, eps_d, kappa_b)}")


## Linearization: eigenvalues and eigenvectors at each fixed point

Every fixed point here is a **saddle** of the full 4D system: 2 stable eigenvalues
(negative real part) and 2 unstable eigenvalues (positive real part), coming in exact
`+/-` pairs. That's the generic signature of the "doubled" formalism (Freidlin-Wentzell /
MPEP): a classically *attracting* fixed point of the original 2-mode system becomes a
*saddle* once you add the conjugate-momentum directions, with eigenvalues `+/-lambda`
for every stable eigenvalue `-lambda` of the original system.

The cell below prints the eigenvalues and shows, for each stable/unstable eigenvector,
which of the 4 coordinates `(A, B, P_A, P_B)` it actually lives in — this is the key
diagnostic for understanding *why* a given connection is or isn't degenerate.


In [ ]:
def eig_info(fp, tol=1e-8):
    """Eigen-decomposition of the linearization at fp.

    Returns eigenvalues w, right eigenvectors V (columns), and index lists for the
    stable / unstable subspaces.
    """
    J = jac(np.asarray(fp, dtype=float), g, kappa_b)
    w, V = np.linalg.eig(J)
    s_idx = [i for i in range(4) if w[i].real < -tol]
    u_idx = [i for i in range(4) if w[i].real > tol]
    return w, V, s_idx, u_idx


def stable_unstable_rows(fp, tol=1e-8):
    """Left-eigenvector rows (dual basis) spanning the stable / unstable subspaces.

    Ws @ (y - fp) == 0  <=>  y - fp has no component along the stable eigenvectors
                              (i.e. y sits on the *unstable* manifold, to linear order).
    Wu @ (y - fp) == 0  <=>  y - fp has no component along the unstable eigenvectors
                              (i.e. y sits on the *stable* manifold, to linear order).
    """
    w, V, s_idx, u_idx = eig_info(fp, tol)
    Vinv = np.linalg.inv(V)
    return Vinv[s_idx].real, Vinv[u_idx].real, w, V


def print_eig_table(name, fp):
    w, V, s_idx, u_idx = eig_info(fp)
    print(f"--- {name} = {np.round(fp, 4)} ---")
    for i in range(4):
        kind = 'stable  ' if i in s_idx else ('unstable' if i in u_idx else 'center? ')
        vec = np.round(V[:, i].real, 3)
        print(f"  lambda = {w[i].real:+.5f}  [{kind}]   eigvec (A,B,PA,PB) = {vec}")
    print()

for name, fp in [('fp_init', fp_init), ('fp_swap', fp_swap),
                  ('fp_fin', fp_fin), ('fp_saddle', fp_saddle)]:
    print_eig_table(name, fp)


## Visualizing the linearization

Both `fp_init` and `fp_swap` have `B = P_B = 0`, so the vector field restricted to the
`(A, P_A)` plane (holding `B = P_B = 0`) is exactly the field these fixed points sit in —
useful for a 2D quiver plot, even though the true connecting trajectory will bulge out of
this plane in `B, P_B` along the way (see the 3rd panel below).

Arrows mark the eigenvectors at each fixed point, projected onto `(A, P_A)`: **blue** for
stable, **red** for unstable.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))

# quiver of the (A, P_A) slice at B = P_B = 0
lim = 1.3 * A0
grid = np.linspace(-lim, lim, 25)
AA, PPA = np.meshgrid(grid, grid)
dA = np.zeros_like(AA)
dPA = np.zeros_like(AA)
for i in range(AA.shape[0]):
    for j in range(AA.shape[1]):
        d = vf([AA[i, j], 0.0, PPA[i, j], 0.0], g, eps_d, kappa_b)
        dA[i, j], dPA[i, j] = d[0], d[2]
speed = np.hypot(dA, dPA)
ax.quiver(AA, PPA, dA / speed, dPA / speed, speed, cmap='Greys', pivot='mid', alpha=0.6)

for name, fp, color in [('fp_init', fp_init, 'C0'), ('fp_swap', fp_swap, 'C1')]:
    ax.scatter(fp[0], fp[2], color=color, s=80, zorder=5, label=name)
    w, V, s_idx, u_idx = eig_info(fp)
    for i in s_idx:
        v = V[[0, 2], i].real
        v = v / np.linalg.norm(v) * 0.15 * A0
        ax.annotate('', xy=(fp[0] + v[0], fp[2] + v[1]), xytext=(fp[0], fp[2]),
                    arrowprops=dict(arrowstyle='-|>', color='tab:blue', lw=2))
    for i in u_idx:
        v = V[[0, 2], i].real
        v = v / np.linalg.norm(v) * 0.15 * A0
        ax.annotate('', xy=(fp[0] + v[0], fp[2] + v[1]), xytext=(fp[0], fp[2]),
                    arrowprops=dict(arrowstyle='-|>', color='tab:red', lw=2))

ax.set_xlabel('A')
ax.set_ylabel(r'$P_A$')
ax.set_title('Phase slice (B=P_B=0): flow + eigenvectors\n'
             'blue = stable, red = unstable (projected onto A, P_A)')
ax.legend(loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()


## Why the direct `fp_init` -> `fp_fin` connection is hard (context)

`P_A = P_B = 0` is an **exactly invariant** subspace of the full nonlinear dynamics (check:
`dP_A/dt = dP_B/dt = 0` whenever `P_A = P_B = 0`, for any `A, B`). It's just the original,
undoubled 2-mode system. Both `fp_init` and `fp_fin` sit inside it and are genuine attractors
of the classical reduced dynamics there — which is exactly why their doubled-phase-space
*stable* eigenvectors both come out purely within `(A, B)` (zero `P_A, P_B` component, see the
eigenvector table above).

The consequence: the boundary condition "land on `fp_fin`'s stable manifold" is, to linear
order, blind to `A` and `B` entirely — it only pins `P_A, P_B -> 0`, which is true
*everywhere* in that invariant plane, including at `fp_init` itself. `solve_bvp` exploits
this and reports a "converged" trajectory that never actually leaves `fp_init`.

`fp_swap = (0, 0, A0, 0)` doesn't have this problem: its stable manifold lives in the
`(P_A, P_B)` plane instead, which is transverse to `fp_init`'s stable manifold. That
boundary condition is genuinely sensitive to the difference between the two points, and the
BVP below converges to a real, non-trivial trajectory.


In [ ]:
def make_bc(fp_source, fp_target):
    """Boundary conditions for a heteroclinic connection fp_source -> fp_target:
      - leave fp_source tangent to its unstable manifold
      - arrive at fp_target tangent to its stable manifold
    """
    Ws_source, _, _, _ = stable_unstable_rows(fp_source)
    _, Wu_target, _, _ = stable_unstable_rows(fp_target)

    def bc(ya, yb):
        c0 = Ws_source @ (ya - fp_source)
        c1 = Wu_target @ (yb - fp_target)
        return np.concatenate([c0, c1])
    return bc


def make_bc_jac(fp_source, fp_target):
    Ws_source, _, _, _ = stable_unstable_rows(fp_source)
    _, Wu_target, _, _ = stable_unstable_rows(fp_target)

    def bc_jac(ya, yb):
        dya = np.vstack([Ws_source, np.zeros_like(Wu_target)])
        dyb = np.vstack([np.zeros_like(Ws_source), Wu_target])
        return dya, dyb
    return bc_jac


## Initial guess

Rather than seeding `solve_bvp` with `fp + eps * v_unstable * exp(lambda * t)` (the
original approach), which blows up numerically whenever `lambda * T` is more than a few
units, use a **bounded** ansatz: a quarter circle `A(t) = A0*cos(phi(t))`,
`P_A(t) = A0*sin(phi(t))`, with `phi` smoothly easing from `0` to `pi/2`. This
automatically starts exactly at `fp_init` and ends exactly at `fp_swap`, stays bounded by
construction, and conserves `A^2 + P_A^2 = A0^2` (matching the fact that both endpoints
sit on that circle). `B, P_B` start at 0; `solve_bvp` will bend those away from zero as
needed.


In [ ]:
def initial_guess(T, N=400):
    t = np.linspace(0, T, N)
    s = t / T
    smoothstep = 3 * s**2 - 2 * s**3         # zero slope at both ends
    phi = smoothstep * np.pi / 2
    A = A0 * np.cos(phi)
    PA = A0 * np.sin(phi)
    B = np.zeros(N)
    PB = np.zeros(N)
    return t, np.vstack([A, B, PA, PB])


t_guess, y0_guess = initial_guess(T=600)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(y0_guess[0], y0_guess[2], '--', color='gray', label='initial guess')
ax.scatter(*fp_init[[0, 2]], color='C0', s=80, zorder=5, label='fp_init')
ax.scatter(*fp_swap[[0, 2]], color='C1', s=80, zorder=5, label='fp_swap')
ax.set_xlabel('A'); ax.set_ylabel(r'$P_A$')
ax.set_title('Initial guess (before solving)')
ax.legend(); ax.set_aspect('equal')
plt.tight_layout()
plt.show()


## How long does the connecting orbit take?

Near `fp_swap` the *slowest* eigenvalue has magnitude only about `0.03` (see the
eigenvalue table above), so the trajectory needs roughly `T >> 1/0.03 ~ 30` just to relax
onto it — and in practice quite a bit more, since the interesting/fast part of the
transition also has to happen somewhere inside that window. The scan below solves the BVP
for a range of total times `T` and plots how close the endpoint actually lands to
`fp_swap`; short `T` isn't enough, but once `T` is long enough the endpoint locks on
essentially exactly.


In [ ]:
bc = make_bc(fp_init, fp_swap)
fun = lambda t, y: vf(y, g, eps_d, kappa_b)

T_scan = [100, 200, 300, 400, 600, 900, 1300]
dist_scan = []
for T in T_scan:
    t_g, y_g = initial_guess(T)
    sol = solve_bvp(fun, bc, t_g, y_g, tol=1e-6, max_nodes=60000, verbose=0)
    dist_scan.append(np.linalg.norm(sol.y[:, -1] - fp_swap))

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(T_scan, dist_scan, 'o-')
ax.set_xlabel('total time T')
ax.set_ylabel('distance of endpoint to fp_swap')
ax.set_title('Convergence to fp_swap vs. integration time')
plt.tight_layout()
plt.show()


In [ ]:
# Final, accurate solve at a T long enough to have converged (see scan above)
T = 1000.0
t_guess, y0_guess = initial_guess(T, N=400)

bc = make_bc(fp_init, fp_swap)
bc_jac = make_bc_jac(fp_init, fp_swap)

sol = solve_bvp(fun, bc, t_guess, y0_guess,
                 fun_jac=lambda t, y: fun_jac(t, y, g, kappa_b),
                 bc_jac=bc_jac,
                 tol=1e-10, max_nodes=200000, verbose=2)

print()
print("start :", sol.y[:, 0], " vs fp_init  =", fp_init)
print("end   :", sol.y[:, -1], " vs fp_swap =", fp_swap)


## Results


In [ ]:
t_plot = np.linspace(sol.x[0], sol.x[-1], 3000)
A, B, PA, PB = sol.sol(t_plot)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].plot(t_plot, A, label='A')
axes[0, 0].plot(t_plot, PA, label=r'$P_A$')
axes[0, 0].set_xlabel('t'); axes[0, 0].legend(); axes[0, 0].set_title('A(t), P_A(t)')

axes[0, 1].plot(t_plot, B, label='B', color='C2')
axes[0, 1].plot(t_plot, PB, label=r'$P_B$', color='C3')
axes[0, 1].set_xlabel('t'); axes[0, 1].legend(); axes[0, 1].set_title('B(t), P_B(t)')

axes[1, 0].plot(A, PA, color='C4')
axes[1, 0].scatter([fp_init[0], fp_swap[0]], [fp_init[2], fp_swap[2]], color='k', zorder=5)
axes[1, 0].set_xlabel('A'); axes[1, 0].set_ylabel(r'$P_A$'); axes[1, 0].set_title('(A, P_A) plane')
axes[1, 0].set_aspect('equal')

axes[1, 1].plot(B, PB, color='C5')
axes[1, 1].scatter([fp_init[1], fp_swap[1]], [fp_init[3], fp_swap[3]], color='k', zorder=5)
axes[1, 1].set_xlabel('B'); axes[1, 1].set_ylabel(r'$P_B$'); axes[1, 1].set_title('(B, P_B) plane')

plt.suptitle(rf'fp_init $\to$ fp_swap, $g={g:.4f},\ \epsilon_d={eps_d:.4f},\ \kappa_b={kappa_b:.4f}$')
plt.tight_layout()
plt.show()


In [ ]:
# Hamiltonian should stay ~0 along the whole path (both endpoints have H = 0)
Hvals = H_of(sol.sol(t_plot), g, eps_d, kappa_b)

plt.figure(figsize=(6, 3))
plt.plot(t_plot, Hvals)
plt.xlabel('t'); plt.ylabel('H(t)')
plt.title(f'max |H| = {np.max(np.abs(Hvals)):.2e}')
plt.tight_layout()
plt.show()


## Next steps

- The mirror-image connection `fp_init -> fp_swap_neg = (0, 0, -A0, 0)` should work the
  same way (just swap `fp_swap` for `fp_swap_neg` in `make_bc`).
- If the physically relevant object is still the full switching path `fp_init -> fp_fin`,
  the natural way to build it (given the degeneracy discussed above) is as a concatenation
  through the classical barrier-top saddle `fp_saddle`, with the second leg
  (`fp_saddle -> fp_fin`) being a plain classical (`P_A = P_B = 0`) relaxation — a simple
  `solve_ivp` integration, not a BVP.
